In [0]:
%sql
--Zone Dimension Table
CREATE OR REPLACE TABLE nyc.default.dim_zones AS
SELECT DISTINCT
    LocationID AS location_id,
    Borough AS borough,
    Zone AS zone_name,
    service_zone
FROM nyc.default.bronze_taxi_zones;

-- Date Dimension Table
CREATE OR REPLACE TABLE nyc.default.dim_date AS
SELECT DISTINCT
    CAST(DATE_FORMAT(pickup_datetime, 'yyyyMMdd') AS INT) AS date_key,
    CAST(pickup_datetime AS DATE) AS full_date,
    YEAR(pickup_datetime) AS year,
    MONTH(pickup_datetime) AS month,
    DAY(pickup_datetime) AS day_of_month,
    DAYOFWEEK(pickup_datetime) AS day_of_week,
    DATE_FORMAT(pickup_datetime, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(pickup_datetime) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM nyc.default.silver_fhv_trips;

-- Hour Dimension Table
CREATE OR REPLACE TABLE nyc.default.dim_hour AS
SELECT DISTINCT
    HOUR(pickup_datetime) AS hour_key,
    CONCAT(LPAD(HOUR(pickup_datetime), 2, '0'), ':00') AS hour_label,
    CASE 
        WHEN HOUR(pickup_datetime) BETWEEN 6 AND 11 THEN 'Morning'
        WHEN HOUR(pickup_datetime) BETWEEN 12 AND 16 THEN 'Afternoon'
        WHEN HOUR(pickup_datetime) BETWEEN 17 AND 21 THEN 'Evening'
        ELSE 'Night'
    END AS time_of_day
FROM nyc.default.silver_fhv_trips;

In [0]:
%sql
CREATE OR REPLACE TABLE nyc.default.fact_trips AS
SELECT 
    dispatching_base_num,
    CAST(DATE_FORMAT(pickup_datetime, 'yyyyMMdd') AS INT) AS pickup_date_key,
    HOUR(pickup_datetime) AS pickup_hour_key,
    PUlocationID AS pickup_location_id,
    DOlocationID AS dropoff_location_id,
    pickup_datetime,
    dropOff_datetime,
    ROUND((UNIX_TIMESTAMP(dropOff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60.0, 2) AS trip_duration_minutes,
    SR_Flag AS is_shared_ride
FROM nyc.default.silver_fhv_trips;

In [0]:
%sql
CREATE OR REPLACE TABLE nyc.default.gold_hourly_zone_summary AS
SELECT 
    d.full_date,
    h.hour_label,
    h.time_of_day,
    z_pu.borough AS pickup_borough,
    z_pu.zone_name AS pickup_zone,
    z_do.zone_name AS dropoff_zone,
    COUNT(*) AS total_trips,
    ROUND(AVG(f.trip_duration_minutes), 2) AS avg_duration_minutes,
    SUM(f.is_shared_ride) AS total_shared_rides
FROM nyc.default.fact_trips f
JOIN nyc.default.dim_date d ON f.pickup_date_key = d.date_key
JOIN nyc.default.dim_hour h ON f.pickup_hour_key = h.hour_key
JOIN nyc.default.dim_zones z_pu ON f.pickup_location_id = z_pu.location_id
JOIN nyc.default.dim_zones z_do ON f.dropoff_location_id = z_do.location_id
GROUP BY 
    d.full_date,
    h.hour_label,
    h.time_of_day,
    z_pu.borough,
    z_pu.zone_name,
    z_do.zone_name;

In [0]:
%sql
-- Example: Top 3 busiest pickup zones per borough using Window Functions
WITH zone_rankings AS (
    SELECT 
        pickup_borough,
        pickup_zone,
        SUM(total_trips) AS total_trips,
        DENSE_RANK() OVER (PARTITION BY pickup_borough ORDER BY SUM(total_trips) DESC) as rank
    FROM nyc.default.gold_hourly_zone_summary
    WHERE pickup_borough != 'Unknown'
    GROUP BY pickup_borough, pickup_zone
)
SELECT pickup_borough, pickup_zone, total_trips
FROM zone_rankings
WHERE rank <= 3
ORDER BY pickup_borough, rank;

In [0]:
%sql
-- Which zones and hours have the most cancellations or longest waits? 
WITH hourly_zone_stats AS (
    SELECT 
        z_pu.borough AS pickup_borough,
        z_pu.zone_name AS pickup_zone,
        h.hour_label,
        COUNT(*) AS total_trips,
        ROUND(AVG(f.trip_duration_minutes), 2) AS avg_duration_minutes,
        DENSE_RANK() OVER (
            PARTITION BY z_pu.borough 
            ORDER BY AVG(f.trip_duration_minutes) DESC
        ) AS rank_within_borough
    FROM nyc.default.fact_trips f
    JOIN nyc.default.dim_zones z_pu ON f.pickup_location_id = z_pu.location_id
    JOIN nyc.default.dim_hour h ON f.pickup_hour_key = h.hour_key
    WHERE z_pu.borough != 'Unknown'
    GROUP BY z_pu.borough, z_pu.zone_name, h.hour_label
    HAVING COUNT(*) > 50 -- Exclude low-sample outliers
)
SELECT 
    pickup_borough,
    pickup_zone,
    hour_label,
    total_trips,
    avg_duration_minutes
FROM hourly_zone_stats
WHERE rank_within_borough = 1
ORDER BY avg_duration_minutes DESC
LIMIT 10;

In [0]:
%sql
--Where are trip durations / utilization lowest
WITH zone_metrics AS (
    SELECT 
        z_pu.borough AS pickup_borough,
        z_pu.zone_name AS pickup_zone,
        COUNT(*) AS total_trips,
        ROUND(AVG(f.trip_duration_minutes), 2) AS avg_duration_minutes,
        PERCENT_RANK() OVER (ORDER BY AVG(f.trip_duration_minutes) ASC) AS duration_percentile
    FROM nyc.default.fact_trips f
    JOIN nyc.default.dim_zones z_pu ON f.pickup_location_id = z_pu.location_id
    WHERE z_pu.borough != 'Unknown'
    GROUP BY z_pu.borough, z_pu.zone_name
    HAVING COUNT(*) >= 100
)
SELECT 
    pickup_borough,
    pickup_zone,
    total_trips,
    avg_duration_minutes,
    ROUND(duration_percentile * 100, 1) AS lowest_duration_percentile
FROM zone_metrics
WHERE duration_percentile <= 0.10
ORDER BY avg_duration_minutes ASC;

In [0]:
%sql
--when demand picks
WITH hourly_demand AS (
    SELECT 
        d.is_weekend,
        h.hour_key,
        h.hour_label,
        h.time_of_day,
        COUNT(*) AS trip_volume
    FROM nyc.default.fact_trips f
    JOIN nyc.default.dim_date d ON f.pickup_date_key = d.date_key
    JOIN nyc.default.dim_hour h ON f.pickup_hour_key = h.hour_key
    GROUP BY d.is_weekend, h.hour_key, h.hour_label, h.time_of_day
),
demand_with_share AS (
    SELECT 
        CASE WHEN is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        hour_label,
        time_of_day,
        trip_volume,
        ROUND(100.0 * trip_volume / SUM(trip_volume) OVER (PARTITION BY is_weekend), 2) AS pct_of_daily_demand,
        DENSE_RANK() OVER (PARTITION BY is_weekend ORDER BY trip_volume DESC) AS peak_rank
    FROM hourly_demand
)
SELECT 
    day_type,
    hour_label,
    time_of_day,
    trip_volume,
    pct_of_daily_demand
FROM demand_with_share
WHERE peak_rank <= 3
ORDER BY day_type, peak_rank;